# Bronze Layer Notebook 
Use this notebook to implement the Bronze ingestion from Unity Catalog Volume raw files into Delta tables.

In [0]:
# Step 1: Import libraries
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

In [0]:
# Step 2: Configure catalog and schema context
# - Set active catalog to main
# - Set active schema to retail
spark.sql("USE CATALOG main")
spark.sql("USE SCHEMA retail")
# - Verify current catalog and schema
display(spark.sql("SELECT current_catalog(), current_schema()"))

In [0]:
# Step 3: Define raw source paths from Bronze volume
bronze_raw_base = "/Volumes/main/retail/bronze/raw"
customers_path = f"{bronze_raw_base}/customers"
order_items_path = f"{bronze_raw_base}/order_items"
orders_path = f"{bronze_raw_base}/orders"
products_path = f"{bronze_raw_base}/products"
watermarks_path = f"{bronze_raw_base}/watermarks.json"


In [0]:
# Step 4: Read raw JSONL for each table and display a sample
customers_raw_df = spark.read.json(customers_path)
products_raw_df = spark.read.json(products_path)
orders_raw_df = spark.read.json(orders_path)
order_items_raw_df = spark.read.json(order_items_path)

display(customers_raw_df)
display(products_raw_df)
display(orders_raw_df)
display(order_items_raw_df)

In [0]:
# Step 5: Normalize _extracted_at, created_at, updated_at columns to timestamp in all DataFrames

def normalize_timestamps(df):
    for col in ['_extracted_at', 'created_at', 'updated_at']:
        if col in df.columns:
            df = df.withColumn(col, F.to_timestamp(col))
    return df

customers_raw_df = normalize_timestamps(customers_raw_df)
products_raw_df = normalize_timestamps(products_raw_df)
orders_raw_df = normalize_timestamps(orders_raw_df)
order_items_raw_df = normalize_timestamps(order_items_raw_df)

In [0]:
display(customers_raw_df)
display(products_raw_df)
display(orders_raw_df)
display(order_items_raw_df)

In [0]:
# Step 7: Write Bronze Delta tables
# - Write table retail.bronze_customers
customers_raw_df.write.format("delta").mode("append").saveAsTable("retail.bronze_customers")
# - Write table retail.bronze_products
products_raw_df.write.format("delta").mode("append").saveAsTable("retail.bronze_products")
# - Write table retail.bronze_orders
orders_raw_df.write.format("delta").mode("append").saveAsTable("retail.bronze_orders")
# - Write table retail.bronze_order_items
order_items_raw_df.write.format("delta").mode("append").saveAsTable("retail.bronze_order_items")

In [0]:
# Step 8: Validate Bronze outputs

# Show 5 sample rows from each bronze table
display(spark.sql("SELECT * FROM bronze_customers LIMIT 5"))
display(spark.sql("SELECT * FROM bronze_products LIMIT 5"))
display(spark.sql("SELECT * FROM bronze_orders LIMIT 5"))
display(spark.sql("SELECT * FROM bronze_order_items LIMIT 5"))

# Check nulls in primary keys
display(spark.sql("SELECT COUNT(*) AS null_count FROM bronze_customers WHERE customer_id IS NULL"))
display(spark.sql("SELECT COUNT(*) AS null_count FROM bronze_products WHERE product_id IS NULL"))
display(spark.sql("SELECT COUNT(*) AS null_count FROM bronze_orders WHERE order_id IS NULL"))
display(spark.sql("SELECT COUNT(*) AS null_count FROM bronze_order_items WHERE order_item_id IS NULL"))



In [0]:
# Compare source-to-bronze counts
display(spark.sql(f"SELECT 'customers' AS table, (SELECT COUNT(*) FROM bronze_customers) AS bronze_count, (SELECT COUNT(*) FROM json.`{customers_path}`) AS source_count"))
display(spark.sql(f"SELECT 'products' AS table, (SELECT COUNT(*) FROM bronze_products) AS bronze_count, (SELECT COUNT(*) FROM json.`{products_path}`) AS source_count"))
display(spark.sql(f"SELECT 'orders' AS table, (SELECT COUNT(*) FROM bronze_orders) AS bronze_count, (SELECT COUNT(*) FROM json.`{orders_path}`) AS source_count"))
display(spark.sql(f"SELECT 'order_items' AS table, (SELECT COUNT(*) FROM bronze_order_items) AS bronze_count, (SELECT COUNT(*) FROM json.`{order_items_path}`) AS source_count"))

In [0]:
# Step 9: Save checkpoints for interview discussion
# - Document decisions: schema inference vs explicit schema
# - Document write mode and partition strategy
# - Document known limitations and next Silver steps